# 10 — Cascade diagnostics, precision-floor robustness, and leakage audit

This notebook does not tune on the test set. It audits the richer snapshot results and the proposed Model B cascade, with special attention to:
- recall subject to a precision floor;
- uncertainty in precision estimates;
- whether each route uses information available at its decision time;
- whether the temporal branch has enough evidence to be enabled.

## Decision-policy clarification

The cascade is a **routing policy**, not an OR ensemble. At any one decision time, only one model route should determine the current risk state. Historical predictions must not automatically accumulate into a permanent positive.

A key timing issue must be explicit: the earlier cascade notebook selects temporal eligibility using `attempts_by_day30` and a latest landmark up to day 30. That is valid only for a decision made at/after the day-30 cutoff. It is **future leakage** if interpreted as a decision made earlier in the 14–30-day interval. This notebook audits that distinction and reports a safe deployment rule.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from statsmodels.stats.proportion import proportion_confint

ROOT_CANDIDATES = [Path('.'), Path('..'), Path('/mnt/data/student_success_assistant')]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p/'data'/'students.csv').exists()), Path.cwd())
OUT = ROOT/'outputs'
RICH = OUT/'rich_snapshot'
CASCADE = OUT/'cascade'
for p in [OUT/'cascade_v2'/'metrics', OUT/'cascade_v2'/'predictions']:
    p.mkdir(parents=True, exist_ok=True)
FLOORS = [0.40, 0.50, 0.60]
print('Project root:', ROOT)

Project root: /Users/abishek/Documents/Projects/student_success_assistant


## 1. Recheck snapshot feature timestamps

Every snapshot feature row should only use source events at or before its declared cutoff. `max_source_event_dt` is an audit field, not a model feature.

In [2]:
feature_audit = []
for h in [7, 14, 30]:
    f = pd.read_csv(RICH/'features'/f'rich_snapshot_{h}d.csv')
    cutoff = pd.to_datetime(f['snapshot_cutoff'], errors='coerce')
    max_source = pd.to_datetime(f['max_source_event_dt'], errors='coerce')
    bad = max_source.notna() & cutoff.notna() & (max_source > cutoff)
    feature_audit.append({
        'horizon_days': h,
        'n_students': f.student_id.nunique(),
        'rows': len(f),
        'max_source_after_cutoff_rows': int(bad.sum()),
        'max_source_after_cutoff_rate': float(bad.mean()),
        'missing_cutoff_rows': int(cutoff.isna().sum()),
        'missing_target_rows': int(f.target_dropout.isna().sum()),
        'duplicate_student_rows': int(f.student_id.duplicated().sum()),
    })
feature_audit = pd.DataFrame(feature_audit)
display(feature_audit)
feature_audit.to_csv(OUT/'cascade_v2'/'metrics'/'snapshot_timestamp_audit.csv', index=False)
assert (feature_audit.max_source_after_cutoff_rows == 0).all(), 'Potential feature-time leakage detected.'
assert (feature_audit.duplicate_student_rows == 0).all(), 'Expected one snapshot row per student and horizon.'

,horizon_days,n_students,rows,max_source_after_cutoff_rows,max_source_after_cutoff_rate,missing_cutoff_rows,missing_target_rows,duplicate_student_rows
0,7,500,500,0,0.0,0,0,0
1,14,500,500,0,0.0,0,0,0
2,30,500,500,0,0.0,0,0,0


## 2. Reconstruct validation-selected snapshot policies

For each horizon/model/floor, choose the validation threshold that maximises recall subject to the point-estimate precision floor. Test metrics are then reported once at that frozen threshold. Test data is not used to choose a threshold or model.

In [3]:
preds = pd.read_csv(RICH/'predictions'/'snapshot_validation_test_probabilities.csv')
def choose_threshold(y, p, floor):
    best = None
    for t in np.unique(np.r_[0.0, p, 1.0]):
        yh = (p >= t).astype(int)
        pr = precision_score(y, yh, zero_division=0)
        re = recall_score(y, yh, zero_division=0)
        if pr >= floor and (best is None or (re, pr, t) > (best[0], best[1], best[2])):
            best = (re, pr, float(t))
    return (best[2], True, best[0], best[1]) if best else (np.nan, False, np.nan, np.nan)

def calc(y, p, t):
    yh = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    pr = precision_score(y, yh, zero_division=0)
    re = recall_score(y, yh, zero_division=0)
    # Wilson interval for precision among the flagged students.
    lo, hi = proportion_confint(int(tp), int(tp+fp), alpha=0.05, method='wilson') if (tp+fp) else (np.nan, np.nan)
    return {'test_precision': pr, 'test_precision_wilson_low': lo, 'test_precision_wilson_high': hi,
            'test_recall': re, 'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'flagged': int(tp+fp), 'test_positive_count': int(np.sum(y)), 'n_test': len(y)}

rows=[]
for h in [7,14,30]:
    for model in sorted(preds.model.unique()):
        v=preds[(preds.horizon_days==h)&(preds.model==model)&(preds.split=='validation')]
        te=preds[(preds.horizon_days==h)&(preds.model==model)&(preds.split=='test')]
        if v.empty or te.empty or v.target_dropout.nunique()<2:
            continue
        for floor in FLOORS:
            t, ok, vr, vp=choose_threshold(v.target_dropout.to_numpy(),v.probability.to_numpy(),floor)
            row={'horizon_days':h,'model':model,'precision_floor':floor,'validation_feasible':ok,
                 'threshold':t,'validation_precision':vp,'validation_recall':vr}
            if ok:
                row.update(calc(te.target_dropout.to_numpy(),te.probability.to_numpy(),t))
                row['test_meets_floor_point_estimate']=row['test_precision'] >= floor
                row['test_meets_floor_wilson_lower_bound']=row['test_precision_wilson_low'] >= floor
            else:
                row.update({'test_precision':np.nan,'test_precision_wilson_low':np.nan,'test_precision_wilson_high':np.nan,
                            'test_recall':np.nan,'TP':np.nan,'FP':np.nan,'FN':np.nan,'TN':np.nan,
                            'flagged':np.nan,'test_positive_count':int(te.target_dropout.sum()),'n_test':len(te),
                            'test_meets_floor_point_estimate':False,'test_meets_floor_wilson_lower_bound':False})
            rows.append(row)
policy = pd.DataFrame(rows)
display(policy.sort_values(['horizon_days','precision_floor','test_recall'],ascending=[True,True,False]))
policy.to_csv(OUT/'cascade_v2'/'metrics'/'snapshot_threshold_robustness.csv',index=False)

,horizon_days,model,precision_floor,validation_feasible,threshold,validation_precision,validation_recall,test_precision,test_precision_wilson_low,test_precision_wilson_high,test_recall,TP,FP,FN,TN,flagged,test_positive_count,n_test,test_meets_floor_point_estimate,test_meets_floor_wilson_lower_bound
6,7,random_forest,0.4,True,0.296964,0.410714,0.851852,0.366667,0.256209,0.493170,0.709677,22,38,9,31,60,31,100,False,False
0,7,hist_gradient_boosting,0.4,True,0.219755,0.400000,0.740741,0.375000,0.260102,0.505946,0.677419,21,35,10,34,56,31,100,False,False
3,7,logistic,0.4,True,0.473507,0.411765,0.777778,0.377358,0.259363,0.511931,0.645161,20,33,11,36,53,31,100,False,False
4,7,logistic,0.5,True,0.749883,0.516129,0.592593,0.500000,0.331541,0.668459,0.483871,15,15,16,54,30,31,100,True,False
7,7,random_forest,0.5,True,0.485869,0.531250,0.629630,0.394737,0.255981,0.552821,0.483871,15,23,16,46,38,31,100,False,False
1,7,hist_gradient_boosting,0.5,True,0.741739,0.571429,0.148148,0.571429,0.250458,0.841780,0.129032,4,3,27,66,7,31,100,True,False
5,7,logistic,0.6,True,0.907782,0.600000,0.333333,0.625000,0.386410,0.815188,0.322581,10,6,21,63,16,31,100,True,False
8,7,random_forest,0.6,True,0.841669,0.666667,0.148148,0.571429,0.250458,0.841780,0.129032,4,3,27,66,7,31,100,False,False
2,7,hist_gradient_boosting,0.6,True,0.791869,0.600000,0.111111,0.400000,0.117621,0.769276,0.064516,2,3,29,66,5,31,100,False,False
9,14,hist_gradient_boosting,0.4,True,0.114479,0.400000,0.888889,0.418182,0.297378,0.549669,0.741935,23,32,8,37,55,31,100,True,False


## 3. What “precision floor satisfied” should mean

The operational constraint is selected on validation. The test point estimate checks generalisation, but a small test set can make precision look deceptively high. The Wilson lower bound is an uncertainty diagnostic; it is not used to tune on test. A cautious deployment policy should ideally demonstrate adequate precision across multiple chronological cutoffs, not just one split.

In [4]:
summary = (policy.groupby(['horizon_days','precision_floor'], as_index=False)
           .agg(validation_feasible_models=('validation_feasible','sum'),
                models_meeting_test_point_floor=('test_meets_floor_point_estimate','sum'),
                models_meeting_test_wilson_floor=('test_meets_floor_wilson_lower_bound','sum'),
                best_test_recall=('test_recall','max'),
                max_test_recall_with_floor=('test_recall', lambda s: s[policy.loc[s.index,'test_meets_floor_point_estimate']].max()
                                            if policy.loc[s.index,'test_meets_floor_point_estimate'].any() else np.nan)))
display(summary)
summary.to_csv(OUT/'cascade_v2'/'metrics'/'snapshot_floor_summary.csv', index=False)

,horizon_days,precision_floor,validation_feasible_models,models_meeting_test_point_floor,models_meeting_test_wilson_floor,best_test_recall,max_test_recall_with_floor
0,7,0.4,3,0,0,0.709677,NaN
1,7,0.5,3,2,0,0.483871,0.483871
2,7,0.6,3,1,0,0.322581,0.322581
3,14,0.4,3,1,0,0.741935,0.741935
4,14,0.5,3,1,0,0.612903,0.580645
5,14,0.6,3,0,0,0.516129,NaN
6,30,0.4,3,2,0,0.967742,0.903226
7,30,0.5,3,1,0,0.838710,0.774194
8,30,0.6,3,0,0,0.516129,NaN


## 4. Audit temporal routing and target timing

The current temporal branch uses a latest landmark up to day 30 and the count of attempts by day 30. That is causally valid **only if the decision is made at day 30**. If the system is meant to route a student during days 14–29, both the attempt count and landmark must be computed as of the actual decision timestamp. Do not use future attempts to decide whether a student qualifies for the temporal branch.

We therefore keep temporal routing disabled in the production cascade until a causal decision-time dataset is constructed and its student-level precision constraint is validated.

In [5]:
seq_path = ROOT/'data'/'curated'/'temporal_attempt_sequences.csv'
students_path = ROOT/'data'/'students.csv'
outcomes_path = ROOT/'data'/'outcome.csv'
seq = pd.read_csv(seq_path)
students = pd.read_csv(students_path)
seq['attempt_dt'] = pd.to_datetime(seq['attempt_dt'], errors='coerce')
seq['signup_dt'] = pd.to_datetime(seq['signup_dt'], errors='coerce')
students['signup_dt'] = pd.to_datetime(students['signup_date'], errors='coerce')
seq = seq.drop(columns=['signup_dt'], errors='ignore').merge(students[['student_id','signup_dt']], on='student_id', how='left', validate='many_to_one')
seq = seq[seq.attempt_dt.notna() & seq.signup_dt.notna() & (seq.attempt_dt >= seq.signup_dt)].copy()
seq['days_from_signup'] = (seq.attempt_dt-seq.signup_dt).dt.total_seconds()/86400
seq30 = seq[seq.days_from_signup <= 30]
counts30 = seq30.groupby('student_id').size()
latest30 = seq30.sort_values(['student_id','attempt_dt']).groupby('student_id').tail(1)
temporal_audit = pd.DataFrame([{
    'students_with_any_attempt_by_day30': int(counts30.size),
    'students_with_more_than_3_attempts_by_day30': int((counts30 > 3).sum()),
    'students_whose_latest_attempt_by_day30_is_before_day14': int((latest30.set_index('student_id').days_from_signup < 14).sum()),
    'interpretation': 'Using day-30 counts/latest landmarks for a decision made before day 30 leaks future activity.'
}])
display(temporal_audit)
temporal_audit.to_csv(OUT/'cascade_v2'/'metrics'/'temporal_routing_timing_audit.csv', index=False)

,students_with_any_attempt_by_day30,students_with_more_than_3_attempts_by_day30,students_whose_latest_attempt_by_day30_is_before_day14,interpretation
0,499,191,127,Using day-30 counts/latest landmarks for a dec...


## 5. Recommended safe cascade v2

Until the temporal branch is rebuilt at actual decision times, use a conservative policy:

- `<7 days`: defer / no prediction.
- `7–<14 days`: day-7 snapshot model, with a frozen threshold selected on validation.
- `14–<30 days`: day-14 snapshot model. Do not route to temporal based on attempts that may occur later in this interval.
- `>=30 days`: day-30 snapshot model.
- Temporal branch: shadow mode only until latest landmark, attempt-count eligibility, and inactivity features are all computed as of the same decision time; then tune the **student-level** threshold on validation.

This does not mean temporal models are inherently worse. It means the current temporal routing implementation is not yet causally aligned with a dynamic 14–30-day decision policy.

Do not OR model predictions. Each decision time has one active route. If a cumulative alert policy is required, evaluate it as a separate policy and jointly tune it on validation.

In [6]:
safe_policy = pd.DataFrame([
    {'age_since_signup':'<7 days','route':'defer','uses_future_information':False,'status':'no alert by design'},
    {'age_since_signup':'7–<14 days','route':'7-day snapshot','uses_future_information':False,'status':'enable only if validation policy feasible; test floor currently unstable'},
    {'age_since_signup':'14–<30 days','route':'14-day snapshot','uses_future_information':False,'status':'recommended interim route; temporal branch shadow-only'},
    {'age_since_signup':'>=30 days','route':'30-day snapshot','uses_future_information':False,'status':'high-recall candidate at 40% floor, high alert burden'},
    {'age_since_signup':'14–<30 days, >3 attempts observed by current decision time','route':'temporal Model B (future candidate)','uses_future_information':False,'status':'not enabled until per-decision-time dataset and student-level precision validation'},
])
display(safe_policy)
safe_policy.to_csv(OUT/'cascade_v2'/'metrics'/'recommended_cascade_policy.csv', index=False)
print('Saved diagnostics to', OUT/'cascade_v2'/'metrics')

,age_since_signup,route,uses_future_information,status
0,<7 days,defer,False,no alert by design
1,7–<14 days,7-day snapshot,False,enable only if validation policy feasible; tes...
2,14–<30 days,14-day snapshot,False,recommended interim route; temporal branch sha...
3,>=30 days,30-day snapshot,False,"high-recall candidate at 40% floor, high alert..."
4,"14–<30 days, >3 attempts observed by current d...",temporal Model B (future candidate),False,not enabled until per-decision-time dataset an...


Saved diagnostics to /Users/abishek/Documents/Projects/student_success_assistant/outputs/cascade_v2/metrics
